In [3]:
import numpy as np

def setup_tridiagonal_matrix(x, y):
    n = len(x)
    h = np.diff(x)  # Step sizes between data points
    
    # Initialize the tridiagonal matrix coefficients
    A = np.zeros(n - 2)  # Subdiagonal
    B = np.zeros(n - 2)  # Main diagonal
    C = np.zeros(n - 2)  # Superdiagonal
    D = np.zeros(n - 2)  # Right-hand side vector
    
    # Populate the tridiagonal matrix coefficients
    for i in range(1, n - 1):
        # Using the standard cubic spline formulas for 'c' coefficients
        A[i - 1] = h[i - 1]                                                     
        B[i - 1] = 2 * (h[i - 1] + h[i])                                        
        C[i - 1] = h[i]                                                         
        D[i - 1] = 3 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])
        
    return A, B, C, D

# Define the known data points
x = np.array([0, 1, 2, 3, 4, 5])
y = np.array([0, 1, 0, 1, 0, 1])

# 1. Get the diagonals
A, B, C, D = setup_tridiagonal_matrix(x, y)

# 2. Construct the full dense matrix from the diagonals
matrix_size = len(B)
M = np.zeros((matrix_size, matrix_size))

for i in range(matrix_size):
    M[i, i] = B[i]            # Main diagonal
    if i > 0:
        M[i, i - 1] = A[i]    # Subdiagonal
    if i < matrix_size - 1:
        M[i, i + 1] = C[i]    # Superdiagonal

# 3. Solve the linear system (M * c = D)
# This gives us the internal 'c' coefficients for our cubic polynomials
c_inner = np.linalg.solve(M, D)

# 4. Apply Natural Boundary Conditions
# A natural spline implies the second derivative (and thus the 'c' coefficient) 
# is 0 at the first and last points.
c_full = np.concatenate(([0], c_inner, [0]))

# --- Print Results ---
print("--- Tridiagonal Arrays ---")
print(f"Subdiagonal   (A): {A}")
print(f"Main diagonal (B): {B}")
print(f"Superdiagonal (C): {C}")
print(f"Right-hand side(D): {D}\n")

print("--- Reconstructed Matrix (M) ---")
print(M)
print("\n--- Solved 'c' Coefficients ---")
print(c_full)

--- Tridiagonal Arrays ---
Subdiagonal   (A): [1. 1. 1. 1.]
Main diagonal (B): [4. 4. 4. 4.]
Superdiagonal (C): [1. 1. 1. 1.]
Right-hand side(D): [-6.  6. -6.  6.]

--- Reconstructed Matrix (M) ---
[[4. 1. 0. 0.]
 [1. 4. 1. 0.]
 [0. 1. 4. 1.]
 [0. 0. 1. 4.]]

--- Solved 'c' Coefficients ---
[ 0.         -2.18181818  2.72727273 -2.72727273  2.18181818  0.        ]
